# 07 — Sensitivity: parcellation resolution

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                    if (p / "analysis" / "lib").exists())
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from joblib import Parallel, delayed

from analysis.lib.data import (load_label, restricted_pool,
                               build_demographics_matrix, identify_binary_targets)
from analysis.lib.stacking import process_stacking_fold
from analysis.lib.cv import pooled_cv

DATA_DIR   = PROJECT_ROOT / "data"
LACUNA_DIR = DATA_DIR / "lacuna" / "METAVCI_PSCI_LACUNA" / "derivatives"
SENS_DIR   = PROJECT_ROOT / "outputs" / "predictive_modeling_sensitivity" / "atlas_resolution"
SENS_DIR.mkdir(parents=True, exist_ok=True)

CLIN_PATH = DATA_DIR / "tabular" / "clinical_data.csv"
assert CLIN_PATH.exists(), f"missing {CLIN_PATH} — see data/README.md"


def _is_schaefer_cortex(col):
    return col.startswith(("7Networks_", "17Networks_"))


REGIMENS = [
    ("regimen_mid",
     "Schaefer-400 (cortex only, from 450 file) + Tian-32 + Nettekoven-32 + xtract",
     [("schaefer2018tian2020parcels450networks17", r"\s+", _is_schaefer_cortex),
      ("tian2020parcels32",                        "\t"),
      ("nettekoven32",                             r"\s+"),
      ("xtract",                                   "\t")]),
    ("regimen_high",
     "Schaefer-Tian-1054 + Nettekoven-default + xtract",
     [("schaefer2018tian2020parcels1054networks17", r"\s+"),
      ("nettekoven",                                r"\s+"),
      ("xtract",                                    "\t")]),
]

In [ ]:
clinical_df = pd.read_csv(CLIN_PATH, index_col=0)
acute_pool = restricted_pool(LACUNA_DIR, clinical_df, all_of=["acuteinfarct"])
clinical = clinical_df.loc[acute_pool]
X_demographics, _, _, _, _ = build_demographics_matrix(clinical, exclude_cohort=False)
binary_targets = identify_binary_targets(clinical)
TARGETS_TO_RUN = binary_targets


def features_for_regimen(label, method, desc, atlases):
    frames = []
    for entry in atlases:
        if len(entry) == 3:
            atlas, sep, col_filter = entry
        else:
            atlas, sep = entry
            col_filter = None
        df = load_label(LACUNA_DIR, label, method, desc, atlases=[(atlas, sep)])
        if col_filter is not None:
            df = df.loc[:, [c for c in df.columns if col_filter(c)]]
        frames.append(df)
    return pd.concat(frames, axis=1).reindex(acute_pool)

In [ ]:
cv = pooled_cv(seed=42)

for regimen_tag, regimen_label, regimen_atlases in REGIMENS:
    out_dir = SENS_DIR / regimen_tag
    if (out_dir / "stacking_scores.csv").exists():
        print(f"[skip] {regimen_tag}: outputs exist")
        continue
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"=== {regimen_tag}: {regimen_label} ===")
    lesion_acute = features_for_regimen("acuteinfarct", "rd",  "damagepct",        regimen_atlases)
    flnm_acute   = features_for_regimen("acuteinfarct", "fnm", "tmap",             regimen_atlases)
    slnm_acute   = features_for_regimen("acuteinfarct", "snm", "disconnectionpct", regimen_atlases)
    print(f"  lesion {lesion_acute.shape[1]} | fLNM {flnm_acute.shape[1]} | sLNM {slnm_acute.shape[1]} parcels")

    stacking_scores_list = []
    for target in TARGETS_TO_RUN:
        y_series = clinical[target]
        valid_mask = y_series.notna()
        y = y_series[valid_mask].values
        X_base = X_demographics[valid_mask.values]
        n_demo = X_base.shape[1]

        view_configs = [
            ("baseline",     X_base),
            ("lesion_acute", np.hstack([X_base, lesion_acute.loc[valid_mask].values])),
            ("flnm_acute",   np.hstack([X_base, flnm_acute.loc[valid_mask].values])),
            ("slnm_acute",   np.hstack([X_base, slnm_acute.loc[valid_mask].values])),
        ]
        print(f"  Stacker on {target} (N={len(y)})")
        fold_results = Parallel(n_jobs=-1)(
            delayed(process_stacking_fold)(i, train, test, view_configs, y, n_demo)
            for i, (train, test) in enumerate(cv.split(X_base, y)))
        for r in fold_results:
            stacking_scores_list.append({
                "Variant": regimen_tag, "Target": target, "Fold": r["Fold"],
                "AUPRC": r["AUPRC"], "AUPRC_train": r["AUPRC_train"]})

    pd.DataFrame(stacking_scores_list).to_csv(out_dir / "stacking_scores.csv", index=False)
    print(f"  saved {out_dir / 'stacking_scores.csv'}")